In [1]:
library(affy)
library(GEOquery)
library(tidyverse)

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: 'generics'


The following objects are masked from 'package:base':

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: 'BiocGenerics'


The following objects are masked from 'package:stats':

    IQR, mad, sd, var, xtabs


The following objects are masked from 'package:base':

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loading required package: Biobase

Welcome to Bioconductor

    Vignettes contain introductory material; view with
    'browseVignettes()'. To cite Bioconductor, see
    'citation("Bioba

In [2]:
# defining datasets
training_set   <- "GSE42568"
validation_set <- c("GSE21653", "GSE20711", "GSE88770")

curr <- validation_set[1]

In [3]:
# reading in .cel files
raw_data <- ReadAffy(celfile.path = paste0("../data/cel_files/", curr))

# RMA normalize data
normalized_data <- rma(raw_data)

# get expression data
normalized_expr <- as.data.frame(exprs(normalized_data))
normalized_expr <- tibble::rownames_to_column(normalized_expr, var = "ID")

# Clean column names - extract just GSM IDs
colnames(normalized_expr) <- gsub("_.*", "", colnames(normalized_expr))

head(normalized_expr)


Warning message:
"replacing previous import 'AnnotationDbi::head' by 'utils::head' when loading 'hgu133plus2cdf'"
Warning message:
"replacing previous import 'AnnotationDbi::tail' by 'utils::tail' when loading 'hgu133plus2cdf'"




Background correcting
Normalizing
Calculating Expression


,ID,GSM540108,GSM540109,GSM540110,GSM540111,GSM540112,GSM540113,GSM540114,GSM540115,GSM540116,⋯,GSM540364,GSM540365,GSM540366,GSM540367,GSM540368,GSM540369,GSM540370,GSM540371,GSM540372,GSM540373
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1007_s_at,10.343735,9.575994,10.205832,10.643041,10.926893,10.230111,11.023431,10.039638,10.411611,⋯,10.144151,10.666997,11.509220,11.144890,10.629969,10.984374,9.974021,10.709648,11.791408,10.051067
2,1053_at,6.021293,6.156694,6.710895,7.513664,6.895131,6.368140,6.756792,6.914100,6.804170,⋯,7.144061,6.724054,8.184394,6.680929,7.147298,8.574534,7.697865,7.261706,8.535019,7.892871
3,117_at,6.085259,6.140420,5.978487,6.108825,6.824378,6.440690,6.391995,6.403060,6.558244,⋯,7.162186,7.514853,6.919280,5.352472,8.725935,7.582938,6.973274,6.842967,6.857275,6.610817
4,121_at,7.813140,7.803163,8.200998,7.848145,7.906104,7.291020,7.174387,7.061505,7.588398,⋯,7.767541,7.706852,7.059599,7.379992,7.162970,8.006837,7.871258,7.801191,7.652344,7.754103
5,1255_g_at,2.904675,3.258130,3.401846,2.934258,2.925637,3.034741,2.883232,3.013251,3.174074,⋯,2.961266,2.846147,2.876281,3.085632,2.981247,3.198020,3.124897,3.110327,3.110366,2.892730
6,1294_at,7.856463,7.734331,9.406229,7.170883,7.775025,8.634346,8.219496,7.967642,8.442348,⋯,8.389826,8.857178,6.838197,8.153096,8.360628,7.378396,8.476990,7.600028,7.086442,8.655083


In [4]:
# map probe IDs to gene symbols
gse <- getGEO(curr, GSEMatrix = TRUE)

Found 1 file(s)

GSE21653_series_matrix.txt.gz



In [5]:
# fetch feature data to get ID - gene symbol mapping
feature_data <- gse[[paste0(curr, "_series_matrix.txt.gz")]]@featureData@data

# subset
feature_data <- feature_data[c(1, 11)]

head(feature_data)

,ID,Gene Symbol
,<chr>,<chr>
1007_s_at,1007_s_at,DDR1 /// MIR4640
1053_at,1053_at,RFC2
117_at,117_at,HSPA6
121_at,121_at,PAX8
1255_g_at,1255_g_at,GUCA1A
1294_at,1294_at,MIR5193 /// UBA7


In [6]:
normalized_expr <- normalized_expr |>
  inner_join(feature_data, by = "ID")

head(normalized_expr)
dim(normalized_expr)

,ID,GSM540108,GSM540109,GSM540110,GSM540111,GSM540112,GSM540113,GSM540114,GSM540115,GSM540116,⋯,GSM540365,GSM540366,GSM540367,GSM540368,GSM540369,GSM540370,GSM540371,GSM540372,GSM540373,Gene Symbol
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,1007_s_at,10.343735,9.575994,10.205832,10.643041,10.926893,10.230111,11.023431,10.039638,10.411611,⋯,10.666997,11.509220,11.144890,10.629969,10.984374,9.974021,10.709648,11.791408,10.051067,DDR1 /// MIR4640
2,1053_at,6.021293,6.156694,6.710895,7.513664,6.895131,6.368140,6.756792,6.914100,6.804170,⋯,6.724054,8.184394,6.680929,7.147298,8.574534,7.697865,7.261706,8.535019,7.892871,RFC2
3,117_at,6.085259,6.140420,5.978487,6.108825,6.824378,6.440690,6.391995,6.403060,6.558244,⋯,7.514853,6.919280,5.352472,8.725935,7.582938,6.973274,6.842967,6.857275,6.610817,HSPA6
4,121_at,7.813140,7.803163,8.200998,7.848145,7.906104,7.291020,7.174387,7.061505,7.588398,⋯,7.706852,7.059599,7.379992,7.162970,8.006837,7.871258,7.801191,7.652344,7.754103,PAX8
5,1255_g_at,2.904675,3.258130,3.401846,2.934258,2.925637,3.034741,2.883232,3.013251,3.174074,⋯,2.846147,2.876281,3.085632,2.981247,3.198020,3.124897,3.110327,3.110366,2.892730,GUCA1A
6,1294_at,7.856463,7.734331,9.406229,7.170883,7.775025,8.634346,8.219496,7.967642,8.442348,⋯,8.857178,6.838197,8.153096,8.360628,7.378396,8.476990,7.600028,7.086442,8.655083,MIR5193 /// UBA7


[1] 54613   268

In [7]:
# For multiple probes → one gene: take average
# For one probe → multiple genes: delete

normalized_expr <- normalized_expr |>
  # Remove rows where gene symbol is empty or maps to multiple genes
  filter(!grepl("///", `Gene Symbol`)) |>
  filter(`Gene Symbol` != "") |>

  # Group by gene symbol and average across probes
  group_by(`Gene Symbol`) |>
  summarise(across(where(is.numeric), mean)) |>
  ungroup() |>
  rename(gene_symbol = `Gene Symbol`)

dim(normalized_expr)
head(normalized_expr)

[1] 21655   267

gene_symbol,GSM540108,GSM540109,GSM540110,GSM540111,GSM540112,GSM540113,GSM540114,GSM540115,GSM540116,⋯,GSM540364,GSM540365,GSM540366,GSM540367,GSM540368,GSM540369,GSM540370,GSM540371,GSM540372,GSM540373
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
A1BG,6.146570,5.632868,7.407916,5.717083,5.667256,5.821602,5.582540,5.678218,6.570997,⋯,5.768699,5.915461,5.332402,5.880671,5.785042,6.160731,5.855461,5.794792,5.933755,5.574898
A1BG-AS1,5.192952,6.270687,5.594629,4.476920,5.047257,5.420414,4.874277,4.710615,5.320223,⋯,4.654715,4.821310,4.533131,4.630453,4.559740,4.400831,4.790866,4.519228,4.606000,4.458904
A1CF,4.136334,4.220885,4.546149,3.887450,3.876345,3.737222,3.619344,3.747788,3.939471,⋯,3.730873,3.814982,3.800488,3.891974,3.807973,3.815930,3.925795,3.902201,4.093958,3.588490
A2M,7.202705,6.328396,8.172368,8.021876,7.659437,8.497944,8.218462,8.664218,8.092863,⋯,8.240868,8.781016,8.344433,7.989748,9.159850,8.391154,7.977827,8.225967,8.341430,8.060287
A2M-AS1,5.378281,4.429702,6.103873,5.039355,4.568131,5.923578,5.995361,6.468561,4.866476,⋯,6.963635,5.482779,4.767218,5.862145,5.259731,4.957411,4.985096,5.038558,4.602791,6.741662
A2ML1,4.452781,4.428543,4.444036,3.829404,3.678200,3.882873,3.741898,3.987309,3.973815,⋯,4.038272,3.880021,4.943441,4.137063,4.040179,3.848122,4.733253,4.720478,4.688482,3.704352


In [8]:
# access the clinical metadata
clinical <- pData(gse[[1]])

colnames(clinical)
head(clinical)

[1] "title"                   "geo_accession"          
 [3] "status"                  "submission_date"        
 [5] "last_update_date"        "type"                   
 [7] "channel_count"           "source_name_ch1"        
 [9] "organism_ch1"            "characteristics_ch1"    
[11] "characteristics_ch1.1"   "characteristics_ch1.2"  
[13] "characteristics_ch1.3"   "characteristics_ch1.4"  
[15] "characteristics_ch1.5"   "characteristics_ch1.6"  
[17] "characteristics_ch1.7"   "characteristics_ch1.8"  
[19] "characteristics_ch1.9"   "characteristics_ch1.10" 
[21] "characteristics_ch1.11"  "characteristics_ch1.12" 
[23] "characteristics_ch1.13"  "molecule_ch1"           
[25] "extract_protocol_ch1"    "label_ch1"              
[27] "label_protocol_ch1"      "taxid_ch1"              
[29] "hyb_protocol"            "scan_protocol"          
[31] "description"             "data_processing"        
[33] "platform_id"             "contact_name"           
[35] "contact_email"           "contact_phone"          
[37] "contact_laboratory"      "contact_department"     
[39] "contact_institute"       "contact_address"        
[41] "contact_city"            "contact_state"          
[43] "contact_zip/postal_code" "contact_country"        
[45] "supplementary_file"      "data_row_count"         
[47] "relation"                "relation.1"             
[49] "age at diagnosis:ch1"    "dfs evt:ch1"            
[51] "dfs time (months):ch1"   "er ihc:ch1"             
[53] "erbb2:ch1"               "histology:ch1"          
[55] "ki67 ihc:ch1"            "molecular subtype:ch1"  
[57] "p53 ihc:ch1"             "pn:ch1"                 
[59] "pr ihc:ch1"              "pt:ch1"                 
[61] "sbr grade:ch1"           "tissue:ch1"

,title,geo_accession,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,characteristics_ch1,⋯,erbb2:ch1,histology:ch1,ki67 ihc:ch1,molecular subtype:ch1,p53 ihc:ch1,pn:ch1,pr ihc:ch1,pt:ch1,sbr grade:ch1,tissue:ch1
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
GSM540108,BC1,GSM540108,Public on May 05 2010,May 04 2010,Aug 28 2018,RNA,1,human breast cancer tumors,Homo sapiens,age at diagnosis: 67,⋯,0,IDC,NA,LuminalB,0,1,0,pT2,2,breast cancer tumor
GSM540109,BC2,GSM540109,Public on May 05 2010,May 04 2010,Aug 28 2018,RNA,1,human breast cancer tumors,Homo sapiens,age at diagnosis: 47,⋯,1,NA,1,LuminalB,1,1,0,NA,3,breast cancer tumor
GSM540110,BC3,GSM540110,Public on May 05 2010,May 04 2010,Aug 28 2018,RNA,1,human breast cancer tumors,Homo sapiens,age at diagnosis: 41,⋯,1,NA,1,LuminalB,1,1,0,NA,3,breast cancer tumor
GSM540111,BC4,GSM540111,Public on May 05 2010,May 04 2010,Aug 28 2018,RNA,1,human breast cancer tumors,Homo sapiens,age at diagnosis: 62,⋯,1,IDC,NA,ERBB2,1,1,0,pT1,3,breast cancer tumor
GSM540112,BC5,GSM540112,Public on May 05 2010,May 04 2010,Aug 28 2018,RNA,1,human breast cancer tumors,Homo sapiens,age at diagnosis: 60,⋯,0,IDC,0,LuminalA,0,1,1,pT2,2,breast cancer tumor
GSM540113,BC6,GSM540113,Public on May 05 2010,May 04 2010,Aug 28 2018,RNA,1,human breast cancer tumors,Homo sapiens,age at diagnosis: 51,⋯,0,IDC,NA,Normal,0,1,0,pT1,2,breast cancer tumor


In [9]:
clinical <- clinical %>%
  mutate(patient_age = gsub("age at diagnosis: ", "", characteristics_ch1)) %>%
  select(-characteristics_ch1) %>%
  mutate(relapse_free_time = as.integer(as.numeric(gsub("dfs time \\(months\\): ", "", `characteristics_ch1.11`)) * 30.44)) %>%
  mutate(t_stage = factor(`pt:ch1`, levels = c("pT1", "pT2", "pT3", "pT4"), ordered = TRUE))

Warning message:
"There was 1 warning in `mutate()`.
ℹ In argument: `relapse_free_time = as.integer(...)`.
Caused by warning:
! NAs introduced by coercion"


In [10]:
# Extract relevant columns
clinical <- clinical[, c(
  "geo_accession",
  "tissue:ch1",
  "patient_age",
  "sbr grade:ch1",
  "t_stage",
  "er ihc:ch1",
  "pn:ch1",
  "dfs evt:ch1",
  "relapse_free_time"
)]

# Rename columns
colnames(clinical) <- c(
  "sample_id",
  "is_tumor",
  "patient_age",
  "tumor_grade",
  "t_stage",
  "er_status",
  "lymph_node_status",
  "relapse_free_event",
  "relapse_free_time"
)

# encode is_tumor as binary
clinical$is_tumor <- ifelse(clinical$is_tumor == "breast cancer tumor", 1, 0)

cols_to_convert <- c("is_tumor", "patient_age", "tumor_grade", "t_stage",
                     "er_status", "lymph_node_status", "relapse_free_event",
                     "relapse_free_time")

clinical[cols_to_convert] <- lapply(clinical[cols_to_convert], as.integer)

Warning message in lapply(clinical[cols_to_convert], as.integer):
"NAs introduced by coercion"
Warning message in lapply(clinical[cols_to_convert], as.integer):
"NAs introduced by coercion"
Warning message in lapply(clinical[cols_to_convert], as.integer):
"NAs introduced by coercion"


Warning message in lapply(clinical[cols_to_convert], as.integer):
"NAs introduced by coercion"
Warning message in lapply(clinical[cols_to_convert], as.integer):
"NAs introduced by coercion"


In [11]:
head(clinical)

,sample_id,is_tumor,patient_age,tumor_grade,t_stage,er_status,lymph_node_status,relapse_free_event,relapse_free_time
,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
GSM540108,GSM540108,1,67,2,2,1,1,1,731
GSM540109,GSM540109,1,47,3,NA,0,1,1,NA
GSM540110,GSM540110,1,41,3,NA,1,1,1,NA
GSM540111,GSM540111,1,62,3,1,0,1,1,1770
GSM540112,GSM540112,1,60,2,2,1,1,1,871
GSM540113,GSM540113,1,51,2,1,0,1,1,301


In [12]:
# save datasets in datasets folder for persistent storage
write.csv(normalized_expr, "../datasets/csv_files/expression_matrix_test_one.csv")
write.csv(clinical, "../datasets/csv_files/clinical_metadata_test_one.csv")